In [ ]:
import os
import subprocess
from dotenv import load_dotenv
from pathlib import Path

from collections import Counter

import pandas as pd

import deeparg
from Bio import SeqIO

from pprint import pprint
from tqdm.auto import tqdm

load_dotenv()

In [ ]:
INPUT_DIR = "/home/ubuntu/projects/biodata/DNA-BERT/data/raw/Wetland Metagenome Assembled Genomes"

In [ ]:
file_extension_counts = Counter(os.path.splitext(f)[1].lower().lstrip(".") for root, dirs, files in os.walk(INPUT_DIR) for f in files)

pprint(file_extension_counts)

| Extension | Meaning                  | Contains                                                                                                       |
| --------- | ------------------------ | -------------------------------------------------------------------------------------------------------------- |
| **.fna**  | FASTA Nucleotide         | Whole genome nucleotide sequences (assembled contigs/chromosomes/scaffolds)                                    |
| **.faa**  | FASTA Amino Acid         | Predicted protein sequences translated from CDS features                                                       |
| **.ffn**  | FASTA Feature Nucleotide | Nucleotide sequences of genes/CDS only (not the entire genome)                                                 |
| **.fsa**  | FASTA Sequence           | Generic FASTA file. Often used during submission workflows. Can contain nucleotide sequences similar to `.fna` |
| **.gff**  | General Feature Format   | Genome annotations: genes, CDS, tRNA, rRNA, coordinates, strands, attributes                                   |
| **.gbk**  | GenBank Format           | Sequence + annotations + metadata in one rich file                                                             |
| **.tbl**  | Feature Table            | NCBI annotation table used during genome submission                                                            |
| **.sqn**  | Sequin File              | Binary submission package used by NCBI's Sequin/BankIt submission system                                       |
| **.err**  | Error Log                | Validation or submission errors generated by annotation/submission software                                    |
| **.log**  | Log File                 | Program execution logs                                                                                         |
| **.tsv**  | Tab-Separated Values     | Annotation summaries, statistics, metadata tables                                                              |
| **.txt**  | Plain Text               | Notes, reports, readme files, miscellaneous output                                                             |


#### Relationship between the important biological files

<pre>
Genome assembly 
    |
    +-- genome.fna   (full nucleotide sequence)
    |
    +-- annotations.gff
    |
    +-- proteins.faa
    |
    +-- genes.ffn
    |
    +-- genome.gbk
</pre>


In [ ]:
faa_files = [os.path.join(root, f) for root, dirs, files in os.walk(INPUT_DIR) for f in files if f.endswith(".faa")]
print("Unique .faa file types:", len(faa_files))

In [ ]:
OUTPUT_DIR = "/home/ubuntu/projects/biodata/DNA-BERT/data/interim/Wetland Metagenome Assembled Genomes/"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

In [ ]:
filed = 0
filed_files = []
completed = 0
total = len(faa_files)

for faa_file in tqdm(faa_files):
    sample = os.path.splitext(os.path.basename(faa_file))[0]
    output_prefix = os.path.join(OUTPUT_DIR, sample)

    cmd = [
        "deeparg",
        "predict",
        "--model",
        "LS",
        "--type",
        "prot",
        "--input",
        faa_file,
        "--output",
        output_prefix,
    ]

    print("Running: " + " ".join(cmd))
    exit_code = subprocess.call(cmd)

    if exit_code != 0:
        filed += 1
        filed_files.append(faa_file)
        print("FAILED: " + faa_file)
    else:
        completed += 1
        print("COMPLETED: " + faa_file)
    print(f"Total: {total}, Completed: {completed}, Failed: {filed}")


In [ ]:
print(f"Processing completed. Total: {total}, Completed: {completed}, Failed: {filed}")
if filed_files:
    print("Failed files:")
    for f in filed_files:
        print(f)

In [ ]:
OUTPUT_DIR = Path(OUTPUT_DIR)
SUMMARY_DIR = OUTPUT_DIR / "_summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def read_deeparg_file(path):
    if path.stat().st_size == 0:
        return pd.DataFrame()

    try:
        return pd.read_csv(path, sep="\t")
    except Exception:
        return pd.read_csv(path, sep="\t", engine="python")

In [ ]:
def find_col(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None

In [ ]:
rows = []
all_arg_rows = []

arg_files = sorted(OUTPUT_DIR.rglob("*.mapping.ARG"))

for arg_file in arg_files:
    sample = arg_file.name.replace(".mapping.ARG", "")
    potential_file = arg_file.with_name(sample + ".mapping.potential.ARG")
    align_file = arg_file.with_name(sample + ".align.daa.tsv")

    arg_df = read_deeparg_file(arg_file)
    potential_df = read_deeparg_file(potential_file) if potential_file.exists() else pd.DataFrame()

    class_col = find_col(arg_df, ["predicted_ARG-class", "predicted_arg-class", "arg_class", "ARG-class"])

    gene_col = find_col(arg_df, ["read_id", "query", "qseqid", "sequence", "gene", "ORF_ID"])

    best_hit_col = find_col(arg_df, ["best-hit", "best_hit", "sseqid", "ARG", "arg"])

    prob_col = find_col(arg_df, ["probability", "prob", "score"])

    n_args = len(arg_df)
    n_potential_args = len(potential_df)

    if class_col and n_args > 0:
        class_counts = arg_df[class_col].value_counts()
        top_class = class_counts.index[0]
        top_class_count = int(class_counts.iloc[0])
        n_classes = int(arg_df[class_col].nunique())
    else:
        top_class = None
        top_class_count = 0
        n_classes = 0

    rows.append(
        {
            "sample": sample,
            "has_ARG": n_args > 0,
            "n_ARGs": n_args,
            "n_potential_ARGs": n_potential_args,
            "n_ARG_classes": n_classes,
            "top_ARG_class": top_class,
            "top_ARG_class_count": top_class_count,
            "arg_file": str(arg_file),
            "potential_arg_file_exists": potential_file.exists(),
            "align_file_exists": align_file.exists(),
        }
    )

    if n_args > 0:
        tmp = arg_df.copy()
        tmp["sample"] = sample
        tmp["arg_file"] = str(arg_file)

        if class_col:
            tmp["deeparg_label"] = tmp[class_col]
        else:
            tmp["deeparg_label"] = "ARG"

        if gene_col:
            tmp["sequence_id"] = tmp[gene_col]
        else:
            tmp["sequence_id"] = tmp.index.astype(str)

        if best_hit_col:
            tmp["deeparg_best_hit"] = tmp[best_hit_col]
        else:
            tmp["deeparg_best_hit"] = None

        if prob_col:
            tmp["deeparg_score"] = tmp[prob_col]
        else:
            tmp["deeparg_score"] = None

        all_arg_rows.append(tmp)


summary_df = pd.DataFrame(rows)

if all_arg_rows:
    all_args_df = pd.concat(all_arg_rows, ignore_index=True)
else:
    all_args_df = pd.DataFrame()


summary_df.to_csv(SUMMARY_DIR / "deeparg_file_summary.tsv", sep="\t", index=False)
all_args_df.to_csv(SUMMARY_DIR / "deeparg_all_positive_ARGs.tsv", sep="\t", index=False)


In [ ]:
# Confusion-matrix-style table:
# rows = genome/bin/sample
# columns = ARG class
# values = count of ARGs in that sample/class
if not all_args_df.empty and "deeparg_label" in all_args_df.columns:
    class_matrix = pd.pivot_table(all_args_df, index="sample", columns="deeparg_label", values="sequence_id", aggfunc="count", fill_value=0)

    class_matrix.to_csv(SUMMARY_DIR / "deeparg_ARG_class_count_matrix.tsv", sep="\t")

    presence_matrix = (class_matrix > 0).astype(int)
    presence_matrix.to_csv(SUMMARY_DIR / "deeparg_ARG_class_presence_matrix.tsv", sep="\t")
else:
    class_matrix = pd.DataFrame()
    presence_matrix = pd.DataFrame()

In [ ]:
# Dataset-level story table
dataset_story = {
    "n_samples_scanned": len(summary_df),
    "n_samples_with_ARGs": int(summary_df["has_ARG"].sum()) if not summary_df.empty else 0,
    "n_samples_without_ARGs": int((~summary_df["has_ARG"]).sum()) if not summary_df.empty else 0,
    "total_ARG_hits": int(summary_df["n_ARGs"].sum()) if not summary_df.empty else 0,
    "total_potential_ARG_hits": int(summary_df["n_potential_ARGs"].sum()) if not summary_df.empty else 0,
    "mean_ARGs_per_sample": float(summary_df["n_ARGs"].mean()) if not summary_df.empty else 0,
    "median_ARGs_per_sample": float(summary_df["n_ARGs"].median()) if not summary_df.empty else 0,
    "max_ARGs_in_one_sample": int(summary_df["n_ARGs"].max()) if not summary_df.empty else 0,
}

pd.DataFrame([dataset_story]).to_csv(SUMMARY_DIR / "deeparg_dataset_story.tsv", sep="\t", index=False)

In [ ]:
# Positive file list
positive_files = summary_df[summary_df["has_ARG"]].copy()
positive_files.to_csv(SUMMARY_DIR / "deeparg_files_with_ARGs.tsv", sep="\t", index=False)

In [ ]:
dfs = {os.path.join(SUMMARY_DIR, f): pd.read_csv(os.path.join(SUMMARY_DIR, f), sep="\t") for f in os.listdir(SUMMARY_DIR)}

In [ ]:
list(dfs.values())[0].T

In [ ]:
for i, (path, df) in enumerate(dfs.items()):
    if i == 0:
        continue
    print(f"File: {os.path.basename(path)}")
    print(f"Shape: {df.shape}")
    if i == 2:  # For the class count matrix, show the full table
        print("non zero entries:", df[df.n_ARGs > 0].shape[0])
        print("non zero potential:", df[df.n_potential_ARGs > 0].shape[0])
        print("non zero non overlapping potential only files:", df[(df.n_ARGs == 0) & (df.n_potential_ARGs > 0)].shape[0])
    # print(df.info())
    display(df.head())

    print("\n\n")

In [ ]:
arg_df = dfs["deeparg_all_positive_ARGs.tsv"]

In [ ]:
arg_df = dfs["/home/ubuntu/projects/biodata/DNA-BERT/data/interim/Wetland Metagenome Assembled Genomes/_summary/deeparg_all_positive_ARGs.tsv"]

In [ ]:
arg_df.head()